# Llama-3-8B-Instruct MMLU Inference
Run this notebook on Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU).
Checkpoints are saved to Google Drive every 100 questions so disconnects don't lose progress.

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Install dependencies
!pip install -q transformers accelerate bitsandbytes pandas pyarrow tqdm

In [ ]:
# Step 3: Set your HuggingFace token
# Get your token from https://huggingface.co/settings/tokens
import os
os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"  # replace this

In [ ]:
# Step 4: Set up paths
# Upload mmlu.parquet to: My Drive/Year 1/215B/ before running this
import pathlib

DRIVE_DIR = pathlib.Path("/content/drive/MyDrive/Year 1/215B")
IN  = DRIVE_DIR / "mmlu.parquet"
OUT = DRIVE_DIR / "llama_instruct_responses.parquet"

assert IN.exists(), f"mmlu.parquet not found at {IN} — upload it to Drive first"
print(f"Input:  {IN}")
print(f"Output: {OUT}")

In [ ]:
# Step 5: Check GPU
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Step 6: Load model and tokenizer (4-bit quantization to fit in Colab T4 RAM)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ["HF_TOKEN"], padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit (this may take a few minutes)...")
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=os.environ["HF_TOKEN"],
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model loaded.")

In [ ]:
# Step 7: Run inference with checkpointing
import pandas as pd
from tqdm.notebook import tqdm

ANSWER_TOKENS = ["A", "B", "C", "D"]
BATCH_SIZE = 8
SAVE_EVERY = 100

def build_prompt(row):
    choices = row["choices"]
    user_msg = (
        f"Question: {row['question']}\n\n"
        f"Choices:\nA. {choices[0]}\nB. {choices[1]}\nC. {choices[2]}\nD. {choices[3]}"
    )
    messages = [
        {"role": "system", "content": "Answer the following multiple choice question with a single letter: A, B, C, or D."},
        {"role": "user", "content": user_msg},
    ]
    # Pre-fill the assistant turn with "The answer is " so we score A/B/C/D
    # at the right position. Without this, Llama-Instruct puts most probability
    # on starting an explanation ("The", "Since", etc.) rather than a bare letter.
    base = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return base + "The answer is "

def get_answer_token_ids(tokenizer):
    ids = {}
    for tok in ANSWER_TOKENS:
        id_no_space = tokenizer.encode(tok, add_special_tokens=False)[0]
        id_with_space = tokenizer.encode(f" {tok}", add_special_tokens=False)[0]
        ids[tok] = (id_no_space, id_with_space)
    return ids

def query_batch(model, tokenizer, prompts, token_ids):
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    last_logits = outputs.logits[:, -1, :]
    log_probs = torch.log_softmax(last_logits, dim=-1).cpu()
    results = []
    for i in range(len(prompts)):
        row = {}
        for tok, (id1, id2) in token_ids.items():
            row[f"logprob_{tok}"] = max(log_probs[i, id1].item(), log_probs[i, id2].item())
        predicted = max(row, key=row.get).replace("logprob_", "")
        results.append({**row, "predicted": predicted})
    return results

# Load data, skip already-completed questions
df = pd.read_parquet(IN)
if OUT.exists():
    done = pd.read_parquet(OUT)["question_id"].tolist()
    df = df[~df["question_id"].isin(done)]
    print(f"Resuming: {len(df):,} questions remaining")
else:
    print(f"Starting fresh: {len(df):,} questions")

token_ids = get_answer_token_ids(tokenizer)
results = []

for i in tqdm(range(0, len(df), BATCH_SIZE)):
    batch = df.iloc[i : i + BATCH_SIZE]
    prompts = [build_prompt(row) for _, row in batch.iterrows()]
    batch_results = query_batch(model, tokenizer, prompts, token_ids)
    for j, (_, row) in enumerate(batch.iterrows()):
        res = batch_results[j]
        res["question_id"] = row["question_id"]
        res["correct"] = res["predicted"] == ANSWER_TOKENS[row["answer"]]
        results.append(res)

    if len(results) >= SAVE_EVERY:
        out_df = pd.DataFrame(results)
        if OUT.exists():
            out_df = pd.concat([pd.read_parquet(OUT), out_df], ignore_index=True)
        out_df.to_parquet(OUT, index=False)
        results = []

if results:
    out_df = pd.DataFrame(results)
    if OUT.exists():
        out_df = pd.concat([pd.read_parquet(OUT), out_df], ignore_index=True)
    out_df.to_parquet(OUT, index=False)

print(f"Done. {len(pd.read_parquet(OUT)):,} total responses saved to {OUT}")

In [ ]:
# Step 7b: Tokenization diagnostic — run this BEFORE the full inference
# Verifies that we're scoring the right tokens and the model outputs what we expect.

import pandas as pd
import torch

df = pd.read_parquet(IN)
row = df.iloc[0]

# Build prompt for the first question
prompt = build_prompt(row)
print("=== PROMPT ===")
print(prompt)
print()

# Check token IDs for A/B/C/D
token_ids = get_answer_token_ids(tokenizer)
print("=== TOKEN IDs ===")
for tok, (id_no_space, id_with_space) in token_ids.items():
    print(f"  '{tok}' (no space): id={id_no_space}, decoded='{tokenizer.decode([id_no_space])}'")
    print(f"  ' {tok}' (space):    id={id_with_space}, decoded='{tokenizer.decode([id_with_space])}'")

# Run inference on the single example
inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model(**inputs)
last_logits = outputs.logits[0, -1, :]
log_probs = torch.log_softmax(last_logits, dim=-1).cpu()

# What token did the model actually want to generate?
top5_ids = log_probs.topk(10).indices.tolist()
print("\n=== TOP-10 TOKENS THE MODEL WOULD GENERATE ===")
for tid in top5_ids:
    print(f"  id={tid:6d}  logprob={log_probs[tid]:.3f}  token='{tokenizer.decode([tid])}'")

# What logprobs does our scoring assign?
print("\n=== OUR SCORED LOGPROBS ===")
correct_answer = ["A", "B", "C", "D"][row["answer"]]
for tok, (id1, id2) in token_ids.items():
    lp = max(log_probs[id1].item(), log_probs[id2].item())
    print(f"  logprob_{tok} = {lp:.3f}  {'← correct answer' if tok == correct_answer else ''}")

predicted = max({tok: max(log_probs[id1].item(), log_probs[id2].item())
                 for tok, (id1, id2) in token_ids.items()},
                key=lambda t: max(log_probs[token_ids[t][0]].item(),
                                  log_probs[token_ids[t][1]].item()))
print(f"\nOur predicted: {predicted}  |  Correct: {correct_answer}")
print(f"Question: {row['question'][:100]}...")
print(f"Choices: {row['choices']}")